# RoBERTa: A Robustly Optimized BERT Pretraining Approach

RoBERTa, developed by Facebook AI, is an improved version of BERT (Bidirectional Encoder Representations from Transformers). The main advancements and new features introduced in RoBERTa compared to BERT are:

## 1. Training Data and Duration

RoBERTa uses a significantly larger dataset (160GB) compared to BERT's original dataset (16GB). The larger dataset includes a more diverse range of texts, allowing the model to learn more generalizable patterns. Mathematically, this means that RoBERTa can capture more complex statistical properties of the language [1, p. 3].

## 2. Dynamic Masking

BERT employs static masking, where 15% of the tokens in the input are masked out once and the same masks are used throughout the training. In contrast, RoBERTa uses dynamic masking where the masking pattern is changed in each epoch [1, p. 4].

Let's denote the input sequence by $X = (x_1, x_2, \ldots, x_n)$. For BERT, a fixed subset $M \subseteq \{1, 2, \ldots, n\}$ is chosen where tokens are masked:
$x_i \rightarrow [MASK] \quad \text{for} \quad i \in M$

For RoBERTa, this subset $M$ is recomputed at every epoch:
$X^{(t)} = (x_1^{(t)}, x_2^{(t)}, \ldots, x_n^{(t)})$
where $x_i^{(t)}$ could be the original token, a [MASK] token, or another token depending on the masking strategy at epoch $t$. This dynamic approach ensures the model doesn't overfit to a particular masking pattern and can generalize better [1, p. 4-5].

## 3. Removal of Next Sentence Prediction (NSP) Objective

In BERT, the NSP objective is used in conjunction with MLM. The NSP task requires the model to predict if a pair of sentences $(A, B)$ are contiguous in the corpus. The NSP loss $L_{NSP}$ is defined as:
$L_{NSP} = - \sum_{(A,B) \in D} \left[ y \log p_{\theta}(y|A, B) + (1-y) \log (1 - p_{\theta}(y|A, B)) \right]$
where $y$ is a binary label indicating if $B$ follows $A$ [2, p. 4171].

RoBERTa removes this objective, focusing solely on MLM:
$L_{MLM} = - \sum_{(X, M) \in D} \sum_{i \in M} \log p_{\theta}(x_i | X_{-i})$
This simplification helps streamline training and focuses the model's capacity on the more challenging task of token prediction [1, p. 5].

## 4. Longer Training with Larger Batches

RoBERTa is trained with larger batches and more iterations. Denoting the batch size by $B$ and the number of training steps by $T$, the effective number of token updates is $B \times T$. Larger batch sizes improve the estimation of gradient descent, reducing variance and potentially leading to better convergence properties [1, p. 6].

Mathematically, for the gradient $g_t$ at step $t$:
$g_t = \frac{1}{B} \sum_{i=1}^B \nabla \ell(x_i; \theta)$
where $\ell(x_i; \theta)$ is the loss for the $i$-th example in the batch. A larger $B$ makes $g_t$ a more accurate estimate of the true gradient [3, p. 2].

## 5. Hyperparameter Tuning

RoBERTa benefits from meticulous hyperparameter tuning, adjusting parameters such as learning rate $\alpha$, batch size $B$, and others. Optimizing these parameters can lead to significant performance improvements. For example, the learning rate schedule can be described by:
$\alpha_t = \alpha_0 \cdot \frac{t_0}{\max(t_0, t)}$
where $t_0$ is a warmup period [1, p. 7].

## 6. Variations in Sequence Length

RoBERTa starts training with shorter sequences and gradually increases the length. Suppose $L_t$ is the sequence length at epoch $t$, we can have:
$L_t = \min(L_{\text{max}}, L_0 + k \cdot t)$
where $L_0$ is the initial sequence length, $L_{\text{max}}$ is the maximum sequence length, and $k$ is an increment factor. This curriculum learning strategy helps in managing computational resources and ensures efficient learning at different stages [1, p. 7].

By focusing on these mathematical aspects, RoBERTa achieves better performance by leveraging more data, more sophisticated training strategies, and optimizing hyperparameters more effectively than BERT.

### References

1. Liu, Y., Ott, M., Goyal, N., Du, J., Joshi, M., Chen, D., Levy, O., Lewis, M., Zettlemoyer, L., & Stoyanov, V. (2019). RoBERTa: A Robustly Optimized BERT Pretraining Approach. arXiv preprint arXiv:1907.11692.

2. Devlin, J., Chang, M. W., Lee, K., & Toutanova, K. (2019). BERT: Pre-training of Deep Bidirectional Transformers for Language Understanding. In Proceedings of the 2019 Conference of the North American Chapter of the Association for Computational Linguistics: Human Language Technologies, Volume 1 (Long and Short Papers) (pp. 4171-4186).

3. You, Y., Gitman, I., & Ginsburg, B. (2017). Large Batch Training of Convolutional Networks. arXiv preprint arXiv:1708.03888.

In [1]:
pip install torch==2.0.1 torchvision torchaudio

INFO: pip is looking at multiple versions of torchvision to determine which version is compatible with other requirements. This could take a while.
INFO: pip is still looking at multiple versions of torchvision to determine which version is compatible with other requirements. This could take a while.
INFO: pip is looking at multiple versions of torchaudio to determine which version is compatible with other requirements. This could take a while.
INFO: pip is still looking at multiple versions of torchaudio to determine which version is compatible with other requirements. This could take a while.
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 619.9/619.9 MB 1.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 317.1/317.1 MB 4.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.8/11.8 MB 61.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 21.0/21.0 MB 50.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 849.3/849.3 kB 42.7 MB/s eta 0:00:00

In [2]:
import torch
import torch.nn as nn
import torch.optim as optim
import random

# Constants
VOCAB_SIZE = 1000  # Numbers from 0 to 999
HIDDEN_SIZE = 128
NUM_HEADS = 4
NUM_LAYERS = 2
MAX_SEQ_LEN = 20
BATCH_SIZE = 64
NUM_EPOCHS = 10

class SimpleTransformerBlock(nn.Module):
    def __init__(self, hidden_size, num_heads):
        super().__init__()
        self.attention = nn.MultiheadAttention(hidden_size, num_heads)
        self.norm1 = nn.LayerNorm(hidden_size)
        self.norm2 = nn.LayerNorm(hidden_size)
        self.feed_forward = nn.Sequential(
            nn.Linear(hidden_size, hidden_size * 4),
            nn.ReLU(),
            nn.Linear(hidden_size * 4, hidden_size)
        )

    def forward(self, x):
        attn_output, _ = self.attention(x, x, x)
        x = self.norm1(x + attn_output)
        ff_output = self.feed_forward(x)
        x = self.norm2(x + ff_output)
        return x

class SimpleRoBERTa(nn.Module):
    def __init__(self, vocab_size, hidden_size, num_heads, num_layers):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, hidden_size)
        self.transformer_blocks = nn.ModuleList([
            SimpleTransformerBlock(hidden_size, num_heads) for _ in range(num_layers)
        ])
        self.fc = nn.Linear(hidden_size, vocab_size)

    def forward(self, x):
        x = self.embedding(x)
        for block in self.transformer_blocks:
            x = block(x)
        return self.fc(x)

def dynamic_masking(input_ids, mask_prob=0.15):
    mask = torch.rand(input_ids.shape, device=input_ids.device) < mask_prob
    masked_input = input_ids.clone()
    masked_input[mask] = VOCAB_SIZE - 1  # Assume last token is [MASK]
    return masked_input, mask

def generate_synthetic_data(batch_size, seq_len, vocab_size):
    # Generate sequences of numbers
    return torch.randint(0, vocab_size - 1, (batch_size, seq_len))

def main():
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    print(f"Using device: {device}")

    model = SimpleRoBERTa(VOCAB_SIZE, HIDDEN_SIZE, NUM_HEADS, NUM_LAYERS).to(device)

    # Use torch.compile() for improved performance
    model = torch.compile(model)

    optimizer = optim.Adam(model.parameters(), lr=1e-3)
    criterion = nn.CrossEntropyLoss()

    for epoch in range(NUM_EPOCHS):
        model.train()
        total_loss = 0
        correct_predictions = 0
        total_predictions = 0

        for batch in range(100):  # 100 batches per epoch
            input_ids = generate_synthetic_data(BATCH_SIZE, MAX_SEQ_LEN, VOCAB_SIZE).to(device)
            masked_input, mask = dynamic_masking(input_ids)

            optimizer.zero_grad()
            outputs = model(masked_input)

            loss = criterion(outputs.view(-1, VOCAB_SIZE), input_ids.view(-1))
            loss.backward()
            optimizer.step()

            total_loss += loss.item()

            # Calculate accuracy
            predictions = outputs.argmax(dim=-1)
            correct_predictions += (predictions[mask] == input_ids[mask]).sum().item()
            total_predictions += mask.sum().item()

        epoch_loss = total_loss / 100
        epoch_accuracy = correct_predictions / total_predictions if total_predictions > 0 else 0
        print(f"Epoch {epoch + 1}/{NUM_EPOCHS}, Loss: {epoch_loss:.4f}, Accuracy: {epoch_accuracy:.4f}")

    # Save the model
    torch.save(model.state_dict(), "simple_roberta_synthetic.pth")
    print("Model saved.")

    # Evaluation
    model.eval()
    with torch.no_grad():
        test_input = generate_synthetic_data(1, MAX_SEQ_LEN, VOCAB_SIZE).to(device)
        masked_test_input, test_mask = dynamic_masking(test_input, mask_prob=0.2)
        test_output = model(masked_test_input)
        predictions = test_output.argmax(dim=-1)

        print("\nExample prediction:")
        print("Original:", test_input[0].cpu().numpy())
        print("Masked:  ", masked_test_input[0].cpu().numpy())
        print("Predicted:", predictions[0].cpu().numpy())

if __name__ == "__main__":
    main()

Using device: cpu


No CUDA runtime is found, using CUDA_HOME='/usr/local/cuda'


Epoch 1/10, Loss: 3.0822, Accuracy: 0.0011
Epoch 2/10, Loss: 1.0943, Accuracy: 0.0011
Epoch 3/10, Loss: 1.0677, Accuracy: 0.0009
Epoch 4/10, Loss: 1.0580, Accuracy: 0.0008
Epoch 5/10, Loss: 1.0489, Accuracy: 0.0006
Epoch 6/10, Loss: 1.0396, Accuracy: 0.0007
Epoch 7/10, Loss: 1.0585, Accuracy: 0.0014
Epoch 8/10, Loss: 1.0396, Accuracy: 0.0009
Epoch 9/10, Loss: 1.0155, Accuracy: 0.0009
Epoch 10/10, Loss: 1.0370, Accuracy: 0.0014
Model saved.

Example prediction:
Original: [869 677 166 639 166 260 240 490 825 409 468 643 603 994 707 588 271 909
 217 769]
Masked:   [869 677 166 999 166 260 240 490 999 409 468 643 603 999 999 588 271 999
 217 769]
Predicted: [869 677 166 780 166 260 240 490 780 409 468 643 603 780 780 588 271 780
 217 769]
